# Predicting Electric Vehicle Purchases

Kaggle Competition

https://www.kaggle.com/competitions/playground-series-s6e9/data


Target: Will_Buy_EV

Metric: Area under the ROC curve

In [97]:
# Import Libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder


In [98]:
# Load the dataset
df = pd.read_csv("train.csv")

## Exploratory Data Analysis (EDA)

In [99]:
df

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,No,Yes,Medium,No
668661,668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,Yes,Yes,Low,No
668662,668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,Yes,Yes,Low,No
668663,668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,Yes,Yes,Low,No


In [100]:
df.isna().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [101]:
df.describe()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level
count,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000
mean,334332.000000,47.039171,84769.266989,32.158298,1.712626,4.960408,7.176314,2.935477
std,193027.103211,12.875448,28648.029042,18.730474,0.729275,3.926843,5.186627,1.429119
min,0.000000,25.000000,30000.000000,5.000000,1.000000,0.000000,0.000000,1.000000
25%,167166.000000,36.000000,67376.000000,17.200000,1.000000,2.000000,3.000000,2.000000
50%,334332.000000,47.000000,84880.000000,33.600000,2.000000,4.000000,6.000000,3.000000
75%,501498.000000,58.000000,102753.000000,47.400000,2.000000,7.000000,10.000000,4.000000
max,668664.000000,69.000000,188549.000000,98.700000,4.000000,14.000000,19.000000,5.000000


In [102]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  str    
 9   City_Type                    668665 non-null  str    
 10  Current_Car_Type             668665 non-null  str    
 11  Home_Charging_Possible       668665 non-null  str    
 12  Subsidy_Available            668665 non-null  str    
 13  Range_Anxi

Insights:

1. No missing values. No need to drop rows.

2. Can drop id. Not useful data to determine EV purchase.

3. Scale all numeric columns

4. str columns to encode:

    Gender

    City_Type

    Current_Car_Type

    Home_Charging_Possible

    Subsidy_Available

    Range_Anxiety_Level

    Will_Buy_EV

In [103]:
# Determine encoding strategy for each column
print("Gender")
print(df["Gender"].unique())

print("City_Type")
print(df["City_Type"].unique())

print("Current_Car_Type")
print(df["Current_Car_Type"].unique())

print("Home_Charging_Possible")
print(df["Home_Charging_Possible"].unique())

print("Subsidy_Available")
print(df["Subsidy_Available"].unique())

print("Range_Anxiety_Level")
print(df["Range_Anxiety_Level"].unique())

print("Will_Buy_EV")
print(df["Will_Buy_EV"].unique())

Gender
<StringArray>
['Male', 'Female', 'Other']
Length: 3, dtype: str
City_Type
<StringArray>
['Suburban', 'Rural', 'Urban']
Length: 3, dtype: str
Current_Car_Type
<StringArray>
['Sedan', 'SUV', 'Hatchback', 'Truck']
Length: 4, dtype: str
Home_Charging_Possible
<StringArray>
['Yes', 'No']
Length: 2, dtype: str
Subsidy_Available
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
Range_Anxiety_Level
<StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str
Will_Buy_EV
<StringArray>
['No', 'Yes']
Length: 2, dtype: str


Strategy:

Convert to Boolean:

    Home_Charging_Possible

    Subsidy_Available

    Will_Buy_EV

OneHot-Encoder:

    Gender
    
    City_Type

    Current_Car_Type

Ordinal Encoder:

    Range_Anxiety_Level

## Preprocessing

In [104]:
# Creating a function so the same can be done for Test set, but without the "fit_transform"
def preprocessing(df_pre):

    # ----- Drop id column. Not necessary. -----
    df_pre = df_pre.drop(columns=["id"])

    # ----- Scale numeric columns -----
    scaler = StandardScaler()
    # Grab a list of all numeric columns
    numeric_columns = df_pre.select_dtypes(include="number").columns.tolist()
    # Standardize all numeric columns
    df_pre[numeric_columns] = scaler.fit_transform(df_pre[numeric_columns])

    # Convert yes/no columns to boolean
    df_pre["Home_Charging_Possible"] = df_pre["Home_Charging_Possible"].map({"Yes": 1, "No": 0})
    df_pre["Subsidy_Available"] = df_pre["Subsidy_Available"].map({"Yes": 1, "No": 0})
    df_pre["Will_Buy_EV"] = df_pre["Will_Buy_EV"].map({"Yes": 1, "No": 0})

    # ----- One Hot Encoders -----
    onehot_encoder = OneHotEncoder(sparse_output=False)
    onehot_columns = ["Gender", "City_Type", "Current_Car_Type"]

    for col in onehot_columns:
        # Create encoder for the current column
        col_encoded = onehot_encoder.fit_transform(df_pre[[col]])

        # Create the new OneHot Encoded columns
        col_df = pd.DataFrame(
            col_encoded,
            columns=onehot_encoder.get_feature_names_out([col]),
            index=df_pre.index)

        # Drop the column that is being encoded
        df_pre = df_pre.drop(columns=[col])

        # Add the encoded columns to the dataframe
        df_pre = pd.concat([df_pre, col_df], axis=1)

    # ----- Ordinal Encoders -----
    Range_Anxiety_Level_encoder = OrdinalEncoder(
    categories=[["Low", "Medium", "High"]])

    df_pre["Range_Anxiety_Level"] = Range_Anxiety_Level_encoder.fit_transform(
    df_pre[["Range_Anxiety_Level"]]).ravel()

    return(df_pre)

In [105]:
preprocessed = preprocessing(df)

In [106]:
df

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,No,Yes,Medium,No
668661,668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,Yes,Yes,Low,No
668662,668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,Yes,Yes,Low,No
668663,668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,Yes,Yes,Low,No


In [107]:
preprocessed

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,...,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,1.472636,0.283361,-0.467597,0.394055,-0.499233,-0.033994,-1.354316,1,0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-0.702048,-1.911800,-1.449954,-0.977170,-0.753891,-0.998012,0.744881,1,0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-1.634055,0.335791,0.247816,-0.977170,0.774056,1.508436,1.444613,0,1,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.472636,-0.390577,-0.451580,0.394055,0.264740,0.351613,0.045149,1,0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.540629,-0.937980,0.995261,-0.977170,-0.753891,-0.805209,0.045149,1,0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,-1.323386,-0.477495,-0.254041,0.394055,0.010082,-0.226798,1.444613,0,1,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
668661,-1.168051,1.578495,-1.449954,0.394055,0.010082,-0.612405,-1.354316,1,1,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
668662,1.317301,1.292297,-0.424885,-0.977170,0.010082,-0.226798,-1.354316,1,1,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
668663,0.307627,1.087466,0.616200,0.394055,-1.263206,-1.383620,-1.354316,1,1,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


## Model Selection

## Hyperparameter Tuning

## Final Evaluation